# Mission 06: LLM-as-Judge & Hill-Climbing - 실습 노트북

이 노트북은 여섯 번째 미션을 진행하며 에이전트의 응답을 LLM 판사가 채점하고, 채점 결과와 피드백을 바탕으로 프롬프트를 자동으로 조율하여 성능을 높여가는 Hill-Climbing 최적화 하네스를 설계합니다.

In [ ]:
# 1. 환경 준비
import sys
import os
import json
from dotenv import load_dotenv

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("app"))
load_dotenv(override=True)

from app.utils.llm import get_llm
from langchain_core.messages import SystemMessage, HumanMessage

### [미션 1] Golden Dataset 정의 및 Generator 준비

에이전트가 완수해야 할 목표 질문과 모범 가이드라인을 정의합니다.

In [ ]:
# 1. 평가 기준 데이터셋 (Golden Dataset) 선언
golden_dataset = [
    {
        "query": "파이썬에서 리스트 컴프리헨션을 쓰는 장점이 뭐야?",
        "reference": "가독성이 향상되고 한 줄로 간결한 리스트 생성이 가능하며, 내부 최적화로 인해 일반 append 루프보다 미세하게 성능상 이점이 있다."
    },
    {
        "query": "에이전트 하네스 엔지니어링의 3대 안티패턴에 대해 설명해줘.",
        "reference": "1) 무한 루프 모니터링 누락, 2) 예외 처리 부재(자율성 오남용), 3) 감사 로그 기록 부재가 있다."
    }
]

llm = get_llm(model_name="google_vertexai:gemini-3.5-flash", temperature=0.0)

def generate_answer(query: str, system_prompt: str) -> str:
    """에이전트(Generator)가 시스템 지침과 사용자 질문을 받아 답변을 생성합니다."""
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ])
    return response.content

print("Generator 준비 완료!")

### [미션 2] LLM-as-a-Judge 채점기 구현

생성된 에이전트의 답변을 기준 답안(Reference)과 비교해 1~5점 사이로 채점하고 피드백을 제공하도록 Judge LLM 프롬프트를 완성하세요.

In [ ]:
def evaluate_answer(query: str, answer: str, reference: str) -> dict:
    """
    LLM Judge가 생성된 답변을 모범 답안과 비교하여 채점합니다.
    반환값은 반드시 'score' (1-5 정수)와 'reason' (평가 사유)를 포함하는 JSON 형식이어야 합니다.
    """
    judge_system = (
        "You are an objective AI Judge. Evaluate the given student answer based on the reference answer.\n"
        "Rate the score from 1 (poor/incorrect) to 5 (excellent/perfect).\n"
        "Return your output strictly as a JSON object with keys 'score' and 'reason'. Do not output anything else."
    )
    
    judge_input = (
        f"User Query: {query}\n"
        f"Student Answer: {answer}\n"
        f"Reference Answer: {reference}"
    )
    
    # TODO: llm을 호출하여 평가 결과를 도출하고 JSON 객체로 파싱하세요.
    result = {"score": 1, "reason": "Not evaluated"}
    return result

print("LLM-as-a-Judge 설계 완료!")

### [미션 3] Hill-Climbing 프롬프트 최적화 자동화 루프 구축

이전 채점 점수가 낮으면 프롬프트를 한 단계 피드백하여 보강 지침을 추가하고 재평가를 실시하는 힐클라이밍 최적화 루프를 완성하세요.

In [ ]:
# 초기 엉성한 시스템 프롬프트
current_prompt = "질문에 대해 짧고 대충 답변하세요."

print(f"🚀 [Hill-Climbing] 초기 프롬프트: {current_prompt}")

for iteration in range(2):
    print(f"\n=== Iteration {iteration + 1} ===")
    
    scores = []
    feedbacks = []
    
    # 모든 Golden Dataset 항목에 대해 평가 진행
    for sample in golden_dataset:
        ans = generate_answer(sample["query"], current_prompt)
        eval_res = evaluate_answer(sample["query"], ans, sample["reference"])
        
        scores.append(eval_res.get("score", 1))
        feedbacks.append(eval_res.get("reason", ""))
        
    avg_score = sum(scores) / len(scores)
    print(f"📊 평균 점수: {avg_score:.2f} / 5.0")
    
    if avg_score >= 4.5:
        print("🎯 목표 점수 도달 완료! 루프를 탈출합니다.")
        break
        
    # TODO: 점수를 개선하기 위해 feedbacks 내용들을 기반으로 프롬프트를 개선 지침으로 치환하는 최적화 로직을 작성하세요.
    # 예: current_prompt 문자열을 좀 더 친절하고 상세히 설명하라는 지침으로 보완
    current_prompt = "질문에 대해 구체적이고 가독성이 향상되도록 상세히 번호 지침을 섞어가며 성실하게 답변하세요."
    print(f"🔄 [Hill-Climbing] 보완된 프롬프트: {current_prompt}")